# IAT 360 – Week 3 Assignment: Audio Classification with ML

**Tasks covered:**
1. Train ML models on the RAVDESS dataset (same as tutorial)
2. Test on the RAVDESS held-out test set
3. Test on personal Actor 26 recordings
4. Compare scaled vs. unscaled features
5. Report metrics and plots for all comparisons

## Step 0 – Setup: clone your forked repository

**Before running this cell:** fork `https://github.com/MIS520/Week3-ClassicML` to your own GitHub account,  
then replace `YOUR_GITHUB_USERNAME` below with your username.

In [ ]:
# Clone YOUR forked repo so all data files are available
!git clone https://github.com/YOUR_GITHUB_USERNAME/Week3-ClassicML.git

# Also clone Actor 26 folder if you uploaded it to GitHub,
# OR just upload the Actor 26 folder manually to Colab via the file panel on the left.

## Step 1 – Imports

In [ ]:
import os
import glob
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import soundfile
import librosa
import librosa.display

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis

print('All imports successful!')

## Step 2 – Feature Extraction Functions

We extract three types of audio features from each file:
- **Chromagram** (12 values) – pitch class energy
- **Mel Spectrogram** (128 values) – frequency content on the mel scale
- **MFCC** (40 values) – compact summary of the audio's spectral shape

These are then stacked into a single 180-feature vector per audio file.

In [ ]:
def feature_chromagram(waveform, sample_rate):
    stft = np.abs(librosa.stft(waveform))
    chroma = np.mean(librosa.feature.chroma_stft(S=stft, sr=sample_rate).T, axis=0)
    return chroma

def feature_melspectrogram(waveform, sample_rate):
    mel = np.mean(librosa.feature.melspectrogram(y=waveform, sr=sample_rate, n_mels=128, fmax=8000).T, axis=0)
    return mel

def feature_mfcc(waveform, sample_rate):
    mfcc = np.mean(librosa.feature.mfcc(y=waveform, sr=sample_rate, n_mfcc=40).T, axis=0)
    return mfcc

def get_features(file_path):
    """Load one audio file and return a 180-dim feature vector."""
    with soundfile.SoundFile(file_path) as audio:
        waveform = audio.read(dtype='float32')
        sr = audio.samplerate

    # Convert stereo to mono if needed
    if waveform.ndim > 1:
        waveform = librosa.to_mono(waveform.T)

    chroma = feature_chromagram(waveform, sr)
    mel    = feature_melspectrogram(waveform, sr)
    mfcc   = feature_mfcc(waveform, sr)

    # Stack into one flat vector: 12 + 128 + 40 = 180 features
    return np.hstack([chroma, mel, mfcc])

print('Feature functions ready.')

## Step 3 – Load RAVDESS Dataset

The RAVDESS filenames encode emotion as the 3rd field (1-indexed) when split by `-`.

In [ ]:
# Emotion label mapping from RAVDESS filename convention
emotions_dict = {
    '01': 'neutral',
    '02': 'calm',
    '03': 'happy',
    '04': 'sad',
    '05': 'angry',
    '06': 'fearful',
    '07': 'disgust',
    '08': 'surprised'
}

def load_ravdess(data_path):
    """Load all RAVDESS wav files, extract features, return X and y arrays."""
    X, y = [], []
    files = glob.glob(os.path.join(data_path, '*', '*.wav'))
    total = len(files)
    for i, file in enumerate(files):
        name = os.path.basename(file)
        emotion_code = name.split('-')[2]
        label = emotions_dict[emotion_code]
        features = get_features(file)
        X.append(features)
        y.append(label)
        print(f'\rProcessed {i+1}/{total} files', end='')
    print()
    return np.array(X), np.array(y)

# Update this path to match where your cloned repo is in Colab
RAVDESS_PATH = '/content/Week3-ClassicML/Audio Data'

X_ravdess, y_ravdess = load_ravdess(RAVDESS_PATH)
print(f'Loaded {X_ravdess.shape[0]} samples, {X_ravdess.shape[1]} features each')

## Step 4 – Explore the RAVDESS Features

Let's look at how many samples per emotion, and the raw feature value ranges.

In [ ]:
# Emotion distribution
labels, counts = np.unique(y_ravdess, return_counts=True)
plt.figure(figsize=(9, 4))
plt.bar(labels, counts, color='steelblue')
plt.title('RAVDESS: Samples per Emotion')
plt.xlabel('Emotion')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('plot_ravdess_distribution.png', dpi=150)
plt.show()
print('Saved: plot_ravdess_distribution.png')

In [ ]:
# Feature value ranges – unscaled
features_df = pd.DataFrame(X_ravdess)

sections = {
    'Chromagram (12)':   features_df.iloc[:, :12],
    'Mel Spectrogram (128)': features_df.iloc[:, 12:140],
    'MFCC (40)':         features_df.iloc[:, 140:180],
}

print('=== RAVDESS Feature Statistics (UNSCALED) ===')
for name, section in sections.items():
    vals = section.values.flatten()
    print(f'{name:30s}  min={vals.min():8.2f}  max={vals.max():8.2f}  mean={vals.mean():8.2f}  std={vals.std():7.2f}')

## Step 5 – Scale the Features

StandardScaler makes each feature have mean=0 and standard deviation=1.  
This is important for distance-based models (SVM, kNN) because large-magnitude features  
would otherwise dominate the distance calculation.

In [ ]:
scaler = StandardScaler()
X_ravdess_scaled = scaler.fit_transform(X_ravdess)

features_scaled_df = pd.DataFrame(X_ravdess_scaled)
print('=== RAVDESS Feature Statistics (STANDARD SCALED) ===')
for name, cols in [('Chromagram (12)', slice(0,12)), ('Mel Spectrogram (128)', slice(12,140)), ('MFCC (40)', slice(140,180))]:
    vals = X_ravdess_scaled[:, cols].flatten()
    print(f'{name:30s}  min={vals.min():8.3f}  max={vals.max():8.3f}  mean={vals.mean():8.3f}  std={vals.std():7.3f}')

## Step 6 – Train/Test Split (RAVDESS)

80% training, 20% test. We use `random_state=42` so results are reproducible.

In [ ]:
# Unscaled split
X_train, X_test_rav, y_train, y_test_rav = train_test_split(
    X_ravdess, y_ravdess, test_size=0.2, random_state=42
)

# Scaled split (same indices)
X_train_sc, X_test_rav_sc, _, _ = train_test_split(
    X_ravdess_scaled, y_ravdess, test_size=0.2, random_state=42
)

print(f'Training samples : {len(X_train)}')
print(f'RAVDESS test samples: {len(X_test_rav)}')

## Step 7 – Load Actor 26 (Your Personal Recordings)

In [ ]:
def load_actor26(folder_path):
    """Load Actor 26 wav files using the same RAVDESS filename convention."""
    X, y = [], []
    files = sorted(glob.glob(os.path.join(folder_path, '*.wav')))
    for file in files:
        name = os.path.basename(file)
        emotion_code = name.split('-')[2]
        label = emotions_dict[emotion_code]
        features = get_features(file)
        X.append(features)
        y.append(label)
        print(f'  {name}  →  {label}')
    return np.array(X), np.array(y)

# Update path if needed
ACTOR26_PATH = '/content/Week3-ClassicML/Actor 26'

X_actor26, y_actor26 = load_actor26(ACTOR26_PATH)

# Scale Actor 26 using the SAME scaler fitted on RAVDESS training data
X_actor26_sc = scaler.transform(X_actor26)

print(f'\nActor 26 samples: {len(X_actor26)}')

### Actor 26 Feature Statistics

Let's compare the raw feature ranges of Actor 26 vs. RAVDESS to understand any differences.

In [ ]:
print('=== Actor 26 Feature Statistics (UNSCALED) ===')
for name, cols in [('Chromagram (12)', slice(0,12)), ('Mel Spectrogram (128)', slice(12,140)), ('MFCC (40)', slice(140,180))]:
    vals = X_actor26[:, cols].flatten()
    print(f'{name:30s}  min={vals.min():8.2f}  max={vals.max():8.2f}  mean={vals.mean():8.2f}  std={vals.std():7.2f}')

print()
print('=== RAVDESS Feature Statistics (UNSCALED, for comparison) ===')
for name, cols in [('Chromagram (12)', slice(0,12)), ('Mel Spectrogram (128)', slice(12,140)), ('MFCC (40)', slice(140,180))]:
    vals = X_ravdess[:, cols].flatten()
    print(f'{name:30s}  min={vals.min():8.2f}  max={vals.max():8.2f}  mean={vals.mean():8.2f}  std={vals.std():7.2f}')

## Step 8 – Define and Train All Models

We train on the RAVDESS 80% training set.  
We test each model on **two** test sets:
- RAVDESS 20% held-out test set
- Actor 26 (your 8 recordings, all used as test — they were never part of training)

In [ ]:
# All 8 models from the tutorial
model_definitions = [
    ('KNN',            KNeighborsClassifier(n_neighbors=5, weights='distance')),
    ('SVM Linear',     SVC(kernel='linear', random_state=42)),
    ('SVM RBF',        SVC(kernel='rbf', C=10, gamma='auto', random_state=42)),
    ('Decision Tree',  DecisionTreeClassifier(random_state=42)),
    ('Random Forest',  RandomForestClassifier(n_estimators=200, random_state=42)),
    ('AdaBoost',       AdaBoostClassifier(random_state=42)),
    ('Naive Bayes',    GaussianNB()),
    ('QDA',            QuadraticDiscriminantAnalysis()),
]

results = []  # will store one dict per (model, scaled/unscaled) combination

for label, model in model_definitions:
    for scale_label, Xtr, Xte_rav, Xte_26 in [
        ('Unscaled', X_train,    X_test_rav,    X_actor26),
        ('Scaled',   X_train_sc, X_test_rav_sc, X_actor26_sc),
    ]:
        # Clone model so we get a fresh one each time
        import copy
        m = copy.deepcopy(model)
        m.fit(Xtr, y_train)

        acc_rav = accuracy_score(y_test_rav, m.predict(Xte_rav))
        acc_26  = accuracy_score(y_actor26,  m.predict(Xte_26))

        results.append({
            'Model':       label,
            'Scaling':     scale_label,
            'RAVDESS Acc': round(acc_rav * 100, 2),
            'Actor26 Acc': round(acc_26  * 100, 2),
            'fitted_model': m,
            'Xte_rav': Xte_rav,
            'Xte_26':  Xte_26,
        })
        print(f'{label:16s} | {scale_label:9s} | RAVDESS: {acc_rav*100:5.1f}%  |  Actor26: {acc_26*100:5.1f}%')

print('\nDone!')

## Step 9 – Summary Table

In [ ]:
# Build a clean summary table (drop stored objects)
summary_df = pd.DataFrame([{k: v for k, v in r.items() if k not in ('fitted_model','Xte_rav','Xte_26')} for r in results])
summary_df = summary_df.sort_values('RAVDESS Acc', ascending=False).reset_index(drop=True)
print(summary_df.to_string(index=False))

## Step 10 – Comparison Bar Chart: Scaled vs. Unscaled

In [ ]:
model_names = [r['Model'] for r in results if r['Scaling'] == 'Unscaled']
rav_unscaled = [r['RAVDESS Acc'] for r in results if r['Scaling'] == 'Unscaled']
rav_scaled   = [r['RAVDESS Acc'] for r in results if r['Scaling'] == 'Scaled']
a26_unscaled = [r['Actor26 Acc'] for r in results if r['Scaling'] == 'Unscaled']
a26_scaled   = [r['Actor26 Acc'] for r in results if r['Scaling'] == 'Scaled']

x = np.arange(len(model_names))
width = 0.2

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(x - 1.5*width, rav_unscaled, width, label='RAVDESS Unscaled', color='steelblue')
ax.bar(x - 0.5*width, rav_scaled,   width, label='RAVDESS Scaled',   color='dodgerblue')
ax.bar(x + 0.5*width, a26_unscaled, width, label='Actor26 Unscaled', color='salmon')
ax.bar(x + 1.5*width, a26_scaled,   width, label='Actor26 Scaled',   color='tomato')

ax.set_xticks(x)
ax.set_xticklabels(model_names, rotation=20, ha='right')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Model Accuracy: RAVDESS vs. Actor 26 — Scaled vs. Unscaled')
ax.legend()
ax.set_ylim(0, 110)
plt.tight_layout()
plt.savefig('plot_model_comparison.png', dpi=150)
plt.show()
print('Saved: plot_model_comparison.png')

## Step 11 – Detailed Metrics for Best Model on RAVDESS Test Set

In [ ]:
# Find the single best model on RAVDESS (scaled version)
best = max((r for r in results if r['Scaling']=='Scaled'), key=lambda r: r['RAVDESS Acc'])
print(f"Best model on RAVDESS: {best['Model']} (Scaled) — {best['RAVDESS Acc']}%")

m = best['fitted_model']
y_pred_rav = m.predict(best['Xte_rav'])

print('\n=== Classification Report (RAVDESS test set) ===')
print(classification_report(y_test_rav, y_pred_rav))

In [ ]:
# Confusion matrix for best model on RAVDESS
emotion_labels = list(emotions_dict.values())
cm = confusion_matrix(y_test_rav, y_pred_rav, labels=emotion_labels)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=emotion_labels)
fig, ax = plt.subplots(figsize=(9, 8))
disp.plot(ax=ax, cmap='Blues', colorbar=False, xticks_rotation=45)
ax.set_title(f"{best['Model']} (Scaled) — RAVDESS Test Set")
plt.tight_layout()
plt.savefig('plot_cm_ravdess_best.png', dpi=150)
plt.show()
print('Saved: plot_cm_ravdess_best.png')

## Step 12 – Detailed Metrics for Best Model on Actor 26

In [ ]:
# Find best model on Actor 26 (scaled)
best26 = max((r for r in results if r['Scaling']=='Scaled'), key=lambda r: r['Actor26 Acc'])
print(f"Best model on Actor 26: {best26['Model']} (Scaled) — {best26['Actor26 Acc']}%")

m26 = best26['fitted_model']
y_pred_26 = m26.predict(best26['Xte_26'])

print('\nTrue labels:      ', list(y_actor26))
print('Predicted labels: ', list(y_pred_26))

print('\n=== Classification Report (Actor 26) ===')
print(classification_report(y_actor26, y_pred_26, zero_division=0))

In [ ]:
# Confusion matrix for best model on Actor 26
cm26 = confusion_matrix(y_actor26, y_pred_26, labels=emotion_labels)
disp26 = ConfusionMatrixDisplay(confusion_matrix=cm26, display_labels=emotion_labels)
fig, ax = plt.subplots(figsize=(9, 8))
disp26.plot(ax=ax, cmap='Reds', colorbar=False, xticks_rotation=45)
ax.set_title(f"{best26['Model']} (Scaled) — Actor 26 Test Set")
plt.tight_layout()
plt.savefig('plot_cm_actor26_best.png', dpi=150)
plt.show()
print('Saved: plot_cm_actor26_best.png')

## Step 13 – Scaled vs. Unscaled Accuracy: Side-by-Side for Each Test Set

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (title, unscaled_vals, scaled_vals) in zip(axes, [
    ('RAVDESS Test Set', rav_unscaled, rav_scaled),
    ('Actor 26 Test Set', a26_unscaled, a26_scaled),
]):
    x = np.arange(len(model_names))
    ax.bar(x - 0.2, unscaled_vals, 0.4, label='Unscaled', color='steelblue')
    ax.bar(x + 0.2, scaled_vals,   0.4, label='Scaled',   color='tomato')
    ax.set_xticks(x)
    ax.set_xticklabels(model_names, rotation=30, ha='right')
    ax.set_ylabel('Accuracy (%)')
    ax.set_title(title)
    ax.set_ylim(0, 110)
    ax.legend()

plt.suptitle('Scaled vs. Unscaled — Effect on Model Accuracy', fontsize=14)
plt.tight_layout()
plt.savefig('plot_scaled_vs_unscaled.png', dpi=150)
plt.show()
print('Saved: plot_scaled_vs_unscaled.png')

## Step 14 – Feature Distribution: RAVDESS vs. Actor 26

Visualizing how the MFCC distributions differ between RAVDESS and your personal recordings.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

feature_sections = [
    ('Chromagram', slice(0, 12)),
    ('Mel Spectrogram', slice(12, 140)),
    ('MFCC', slice(140, 180)),
]

for ax, (name, cols) in zip(axes, feature_sections):
    rav_vals = X_ravdess[:, cols].flatten()
    a26_vals = X_actor26[:, cols].flatten()
    ax.hist(rav_vals, bins=50, alpha=0.6, label='RAVDESS', color='steelblue', density=True)
    ax.hist(a26_vals, bins=20, alpha=0.6, label='Actor 26', color='tomato', density=True)
    ax.set_title(name)
    ax.set_xlabel('Feature Value')
    ax.set_ylabel('Density')
    ax.legend()

plt.suptitle('Feature Value Distributions: RAVDESS vs. Actor 26', fontsize=13)
plt.tight_layout()
plt.savefig('plot_feature_distributions.png', dpi=150)
plt.show()
print('Saved: plot_feature_distributions.png')

## Step 15 – Save All Results to CSV

In [ ]:
# Final clean summary (drop fitted model objects)
final_df = pd.DataFrame([{k: v for k, v in r.items() if k not in ('fitted_model','Xte_rav','Xte_26')} for r in results])
final_df.to_csv('results_summary.csv', index=False)
print('Saved: results_summary.csv')
final_df.sort_values('RAVDESS Acc', ascending=False).reset_index(drop=True)

## Summary

**What we did:**
- Extracted 180 audio features (12 chroma + 128 mel + 40 MFCC) from all RAVDESS samples
- Trained 8 ML classifiers on 80% of RAVDESS
- Tested each on the 20% RAVDESS hold-out AND on 8 Actor 26 personal recordings
- Repeated all experiments with StandardScaler applied

**Key files saved:**
| File | Description |
|---|---|
| `results_summary.csv` | Accuracy table for all model × scaling combinations |
| `plot_model_comparison.png` | Bar chart: all models, both test sets, scaled vs. unscaled |
| `plot_scaled_vs_unscaled.png` | Side-by-side scaled vs. unscaled per test set |
| `plot_cm_ravdess_best.png` | Confusion matrix for best model on RAVDESS |
| `plot_cm_actor26_best.png` | Confusion matrix for best model on Actor 26 |
| `plot_feature_distributions.png` | Feature distribution comparison |

Use these plots as screenshots in your report!